# 🚨 DSS de Priorización Temprana y Explicabilidad - 112 Castilla y León (Google Colab)

Este notebook automatiza la configuración e inicio de todo el entorno del TFM para poder interactuar con la interfaz del operador 112 desde tu navegador local. 

### **¿Cómo maneja Colab la ejecución de celdas?**
- En Google Colab, **solo una celda puede ejecutarse en primer plano a la vez**.
- Para evitar bloqueos, este notebook lanza los servidores persistentes (**Ollama, FastAPI y Streamlit**) en **segundo plano** de manera asíncrona usando `subprocess.Popen` en Python.
- La **única** celda que bloquea (se ejecuta en primer plano de forma persistente) es la última, que levanta el túnel de Cloudflare para mostrarte la URL pública de acceso en tiempo real.
- Cada celda de inicio incluye comandos de limpieza (`pkill`) para liberar puertos y evitar el error `Address already in use` si decides reiniciar o volver a ejecutar las celdas.

## 1. Clonar Repositorio e Instalar Dependencias

In [ ]:
#@title Configuración de la Rama e Instalación
RAMA = "brian_capa3" #@param {type:"string"}

# Clonar el repositorio del proyecto especificando la rama
!git clone -b {RAMA} https://github.com/jalbfil/tfe_muia_unir_188.git
%cd tfe_muia_unir_188

# Instalar uv para acelerar la instalación de paquetes
!pip install uv

# Instalar las dependencias del monorepo en el entorno global de Colab
!uv pip install --system -e .

# Instalar dependencias adicionales de ejecución
!uv pip install --system faster-whisper streamlit uvicorn httpx

## 2. Instalar e Iniciar Ollama
Instalamos `zstd`, `lshw` y `pciutils` (requeridos para la extracción y detección correcta de GPU), descargamos Ollama, lo iniciamos en segundo plano y descargamos el modelo de Capa 3 (`llama3.1:8b-instruct-q4_K_M`).

In [ ]:
# Liberar el puerto/proceso de Ollama por si ya estaba corriendo
!pkill -f "ollama" || true

# Instalar zstd, lshw y pciutils (necesarios para Ollama y detección de GPU)
print("Instalando dependencias del sistema (zstd, lshw, pciutils)...")
!sudo apt-get update && sudo apt-get install -y zstd lshw pciutils

# Descargar e instalar Ollama en Linux
print("Instalando Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# Levantar Ollama en segundo plano (no bloquea el notebook)
import subprocess
import time

print("Iniciando servicio de Ollama...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Espera para inicialización de puertos

# Descargar el modelo para el explicador de Capa 3 (esta descarga sí se ejecuta en primer plano)
print("Descargando modelo llama3.1 de Ollama...")
!ollama pull llama3.1:8b-instruct-q4_K_M

## 3. Instalar Cloudflare Tunnel (`cloudflared`)
Descargamos e instalamos el binario debian oficial de Cloudflare para crear túneles públicos sin necesidad de registro.

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 4. Iniciar Backend FastAPI
Limpiamos cualquier proceso previo que use uvicorn y lanzamos el servidor FastAPI de fondo.

In [ ]:
# Limpiar procesos previos de uvicorn
!pkill -f "uvicorn" || true

import subprocess
import time

print("Levantando el backend FastAPI en el puerto 8000...")
backend_process = subprocess.Popen(
    ["uvicorn", "backend.api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="src"
)
time.sleep(4)  # Esperar a que el backend inicialice

## 5. Iniciar la Interfaz Streamlit y Crear el Túnel
Lanzamos Streamlit en segundo plano, liberamos túneles previos de Cloudflare y abrimos el nuevo túnel público en primer plano.

**Instrucciones:** Esta celda quedará ejecutándose continuamente (bloqueada). Busca la línea que dice `Your quick tunnel has been created! Visit it at:` y abre ese enlace `trycloudflare.com` en tu navegador.

In [ ]:
# Limpiar ejecuciones previas de Streamlit y Cloudflare
!pkill -f "streamlit" || true
!pkill -f "cloudflared" || true

import subprocess
import time

print("Iniciando la interfaz web en Streamlit...")
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "src/ui/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)
time.sleep(3)

print("Abriendo túnel de Cloudflare... Haz clic en el enlace 'trycloudflare.com' que aparecerá abajo:")
!cloudflared tunnel --url http://localhost:8501